# Retry audit — halaman yang belum berhasil

Notebook ini **lanjutan** dari audit sebelumnya. Fungsinya:
1. Upload 4 hasil kemarin (`audit_findings.csv`, `page_sample_updated.csv`, `excluded_websites.csv`, `website_master.csv`).
2. Otomatis cari baris yang **gagal render** (timeout / connection error) atau **belum pernah dicoba** (dari website yang tadinya di-exclude) — tapi tetap punya `page_url` yang valid.
3. Audit ulang **hanya baris-baris itu** (bukan 170 baris dari awal).
4. Gabungkan hasil baru ke 4 file lama → keluar versi terbaru dari 4 file yang sama.

Baris yang `page_url`-nya memang kosong dari awal (belum ada link-nya) **tetap di-skip**, karena itu bukan masalah render — itu belum ada datanya sama sekali.

## 1. Install Chrome + Selenium

In [ ]:
!wget -q -O /tmp/google-chrome-stable.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!ls -la /tmp/google-chrome-stable.deb
!apt-get update -qq
!apt install -y /tmp/google-chrome-stable.deb
!apt --fix-broken install -y

import subprocess, shutil

chrome_path = shutil.which('google-chrome') or shutil.which('google-chrome-stable')
if chrome_path is None:
    raise RuntimeError(
        "google-chrome tidak ketemu setelah instalasi. Scroll ke atas, baca output "
        "'apt install' di cell ini untuk lihat pesan errornya."
    )

chrome_version = subprocess.run([chrome_path, '--version'], capture_output=True, text=True).stdout.strip()
print("Chrome path:", chrome_path)
print("Chrome version:", chrome_version)
!pip install -q -U selenium

-rw-r--r-- 1 root root 139922812 Aug  1 21:33 /tmp/google-chrome-stable.deb
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'google-chrome-stable' instead of '/tmp/google-chrome-stable.deb'
The following additional packages will be installed:
  at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0 libatk1.0-0
  libatk1.0-data libatspi2.0-0 libvulkan1 libxcomposite1 libxtst6
  mesa-vulkan-drivers session-migration
The following NEW packages will be installed:
  at-spi2-core google-chrome-stable gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk1.0-0 libatk1.0-data libatspi2.0-0 libvulkan1
  libxcomposite1 libxtst6 mesa-vulkan-drivers session-migration
0 upgraded, 12 newly installed, 0 to remove and 131 not upgraded.
Need to get 11

## 2. Upload 4 file hasil audit sebelumnya

In [ ]:
from google.colab import files
import pandas as pd
import io

print("Upload audit_findings.csv, page_sample_updated.csv, excluded_websites.csv, website_master.csv")
uploaded = files.upload()

def load(name_contains):
    for name in uploaded:
        if name_contains in name:
            return pd.read_csv(io.BytesIO(uploaded[name]), dtype=str).fillna('')
    raise FileNotFoundError(f"File mengandung '{name_contains}' tidak ditemukan di upload.")

audit_findings_old = load('audit_findings')
page_sample_old = load('page_sample')
excluded_old = load('excluded_websites')
website_master_old = load('website_master')

print(f"audit_findings lama: {len(audit_findings_old)} baris")
print(f"page_sample lama: {len(page_sample_old)} baris")
print(f"excluded_websites lama: {len(excluded_old)} baris")
print(f"website_master lama: {len(website_master_old)} baris")

Upload audit_findings.csv, page_sample_updated.csv, excluded_websites.csv, website_master.csv


Saving excluded_websites.csv to excluded_websites.csv
Saving page_sample_updated.csv to page_sample_updated.csv
Saving audit_findings.csv to audit_findings (1).csv
Saving website_master (1).csv to website_master (1).csv
audit_findings lama: 4062 baris
page_sample lama: 170 baris
excluded_websites lama: 5 baris
website_master lama: 33 baris


## 3. Cari baris yang perlu di-retry

Gabungkan `page_sample_updated` (baris yang sempat dicoba) dengan `excluded_websites` (baris dari website yang di-skip total, sebagian belum pernah dicoba sama sekali), lalu ambil yang:
- `page_url` tidak kosong, DAN
- `render_status` bukan `'rendered'` (termasuk kosong/NaN = belum pernah dicoba)

In [ ]:
combined = pd.concat([page_sample_old, excluded_old], ignore_index=True)
combined = combined.drop_duplicates(subset='page_id', keep='first')

has_url = combined['page_url'].str.strip() != ''
not_rendered = combined['render_status'].str.strip() != 'rendered'
retry_candidates = combined[has_url & not_rendered].copy()

print(f"Total baris unik: {len(combined)}")
print(f"Baris yang perlu di-retry (punya URL, belum berhasil render): {len(retry_candidates)}")
print(retry_candidates[['page_id', 'website_id', 'page_type', 'render_status']].to_string(index=False))

Total baris unik: 170
Baris yang perlu di-retry (punya URL, belum berhasil render): 14
page_id website_id        page_type                                                                                                 render_status
   P026        W06         Homepage                                                                                                       timeout
   P031        W07         Homepage                                                                                                       timeout
   P032        W07           Profil                                                                                                       timeout
   P034        W07           Konten                                                                                                       timeout
   P035        W07 Interaksi/Kontak                                                                                                       timeout
   P126        W26         Homepage  

## 4. Setup driver (konfigurasi sama seperti audit pertama, timeout dinaikkan)

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException, WebDriverException

VIEWPORT_WIDTH = 1366
VIEWPORT_HEIGHT = 768
PAGE_LOAD_TIMEOUT_SEC = 45  # dinaikkan dari 30s di audit pertama, karena baris ini sebelumnya timeout

def new_driver():
    options = Options()
    import shutil as _shutil
    options.binary_location = _shutil.which("google-chrome") or _shutil.which("google-chrome-stable")
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument(f'--window-size={VIEWPORT_WIDTH},{VIEWPORT_HEIGHT}')
    driver = webdriver.Chrome(options=options)
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SEC)
    return driver

_test_driver = new_driver()
_test_driver.get("https://example.com")
print("Driver OK, judul halaman test:", _test_driver.title)
_test_driver.quit()
print(f"Viewport: {VIEWPORT_WIDTH}x{VIEWPORT_HEIGHT}, timeout: {PAGE_LOAD_TIMEOUT_SEC}s")

Driver OK, judul halaman test: Example Domain
Viewport: 1366x768, timeout: 45s


## 5. Inject axe-core & fungsi parsing WCAG SC (sama seperti sebelumnya)

In [ ]:
import requests

AXE_CORE_URL = "https://cdnjs.cloudflare.com/ajax/libs/axe-core/4.9.1/axe.min.js"
AXE_SOURCE = requests.get(AXE_CORE_URL, timeout=20).text
print(f"axe-core source loaded: {len(AXE_SOURCE)} chars")

def parse_wcag_sc(tags):
    # Ubah tag axe-core seperti 'wcag143' -> '1.4.3'.
    scs = []
    for t in tags:
        if t.startswith('wcag') and t[4:].isdigit() and len(t) > 4:
            digits = t[4:]
            if len(digits) >= 3:
                principle, guideline, sc = digits[0], digits[1], digits[2:]
                scs.append(f"{principle}.{guideline}.{sc}")
    return ";".join(sorted(set(scs)))

def parse_wcag_level(tags):
    if any(t in tags for t in ('wcag2aa', 'wcag21aa', 'wcag22aa')):
        return 'AA'
    if any(t in tags for t in ('wcag2a', 'wcag21a', 'wcag22a')):
        return 'A'
    return ''

AXE_RUN_SCRIPT = """
var callback = arguments[arguments.length - 1];
axe.run(document, {resultTypes: ['violations']}).then(function(results){
    callback(results);
}).catch(function(err){
    callback({error: String(err)});
});
"""


axe-core source loaded: 555031 chars


## 6. Jalankan retry (hanya baris yang gagal)

In [ ]:
import time
import hashlib
import re
from datetime import datetime, timezone

def normalize_selector(selector_list):
    sel = " ".join(selector_list) if isinstance(selector_list, list) else str(selector_list)
    return re.sub(r'\s+', ' ', sel.strip().lower())

def make_signature(website_id, page_id, rule_id, normalized_selector):
    raw = f"{website_id}|{page_id}|{rule_id}|{normalized_selector}"
    return hashlib.sha256(raw.encode('utf-8')).hexdigest()[:16]

existing_finding_ids = set(audit_findings_old['finding_id']) if len(audit_findings_old) else set()
next_finding_num = max([int(fid[1:]) for fid in existing_finding_ids] or [0]) + 1

new_findings = []
new_status_updates = []

driver = new_driver()

for i, row in retry_candidates.iterrows():
    page_id = row['page_id']
    website_id = row['website_id']
    url = row['page_url'].strip()
    tested_at = datetime.now(timezone.utc).isoformat()
    render_status = 'unknown'

    try:
        driver.get(url)
        time.sleep(2)
        render_status = 'rendered'
    except TimeoutException:
        render_status = 'timeout'
    except WebDriverException as e:
        render_status = f'error: {str(e)[:100]}'

    if render_status != 'rendered':
        new_status_updates.append({'page_id': page_id, 'render_status': render_status, 'tested_at': tested_at})
        print(f"[MASIH GAGAL] {page_id} ({website_id}) - {render_status}: {url}")
        continue

    try:
        driver.execute_script(AXE_SOURCE)
        results = driver.execute_async_script(AXE_RUN_SCRIPT)
    except WebDriverException as e:
        new_status_updates.append({'page_id': page_id, 'render_status': f'axe_error: {str(e)[:100]}', 'tested_at': tested_at})
        print(f"[AXE ERROR] {page_id}: {e}")
        continue

    if isinstance(results, dict) and results.get('error'):
        new_status_updates.append({'page_id': page_id, 'render_status': f'axe_error: {results["error"][:100]}', 'tested_at': tested_at})
        continue

    violations = results.get('violations', [])
    for v in violations:
        rule_id = v.get('id', '')
        tags = v.get('tags', [])
        wcag_sc = parse_wcag_sc(tags)
        wcag_level = parse_wcag_level(tags)
        impact_label = v.get('impact', '') or ''
        nodes = v.get('nodes', [])
        for node in nodes:
            selector = normalize_selector(node.get('target', []))
            signature = make_signature(website_id, page_id, rule_id, selector)
            new_findings.append({
                'finding_id': f"F{next_finding_num:06d}",
                'page_id': page_id, 'website_id': website_id, 'rule_id': rule_id,
                'wcag_sc': wcag_sc, 'wcag_level': wcag_level, 'impact_label': impact_label,
                'affected_node_count': len(nodes), 'selector': selector,
                'finding_signature': signature, 'validation_status': 'auto-detected',
            })
            next_finding_num += 1

    new_status_updates.append({'page_id': page_id, 'render_status': 'rendered', 'tested_at': tested_at})
    print(f"[BERHASIL] {page_id} ({website_id}) - {len(violations)} rule gagal")

driver.quit()
print(f"\nRetry selesai. Baris yang sekarang berhasil render: {sum(1 for s in new_status_updates if s['render_status']=='rendered')}/{len(retry_candidates)}")

[MASIH GAGAL] P026 (W06) - timeout: https://dinsos.sumselprov.go.id/
[MASIH GAGAL] P031 (W07) - timeout: https://dinsos.bengkuluprov.go.id/
[MASIH GAGAL] P032 (W07) - timeout: https://dinsos.bengkuluprov.go.id/profil/
[MASIH GAGAL] P034 (W07) - timeout: https://dinsos.bengkuluprov.go.id/informasi-berkala/
[MASIH GAGAL] P035 (W07) - timeout: https://dinsos.bengkuluprov.go.id/kontak-kami/
[BERHASIL] P126 (W26) - 4 rule gagal
[MASIH GAGAL] P136 (W28) - timeout: https://dinsos.sultraprov.go.id/home
[MASIH GAGAL] P137 (W28) - timeout: https://dinsos.sultraprov.go.id/profil/sambutan
[MASIH GAGAL] P139 (W28) - timeout: https://dinsos.sultraprov.go.id/page/berita/0
[MASIH GAGAL] P140 (W28) - timeout: https://dinsos.sultraprov.go.id/page/contact_us
[BERHASIL] P147 (W30) - 3 rule gagal
[MASIH GAGAL] P148 (W30) - timeout: https://dinsosp3apmd.sulbarprov.go.id/page/layanan
[MASIH GAGAL] P149 (W30) - error: Message: unknown error: net::ERR_CONNECTION_REFUSED
  (Session info: chrome=151.0.7922.71)
S

## 7. Gabungkan hasil retry ke 4 file lama

In [ ]:
new_findings_df = pd.DataFrame(new_findings, columns=audit_findings_old.columns if len(audit_findings_old) else
    ['finding_id','page_id','website_id','rule_id','wcag_sc','wcag_level','impact_label',
     'affected_node_count','selector','finding_signature','validation_status'])
audit_findings_new = pd.concat([audit_findings_old, new_findings_df], ignore_index=True)
audit_findings_new = audit_findings_new.drop_duplicates(subset='finding_signature', keep='first')
audit_findings_new.to_csv('audit_findings.csv', index=False)
print(f"audit_findings.csv: {len(audit_findings_old)} -> {len(audit_findings_new)} baris")

status_df = pd.DataFrame(new_status_updates).set_index('page_id') if new_status_updates else pd.DataFrame(columns=['render_status','tested_at']).set_index(pd.Index([], name='page_id'))

page_sample_new = combined.copy().set_index('page_id')
for col in ['render_status', 'tested_at']:
    if col in status_df.columns:
        page_sample_new.loc[status_df.index, col] = status_df[col]
page_sample_new = page_sample_new.reset_index()
page_sample_new.to_csv('page_sample_updated.csv', index=False)
print(f"page_sample_updated.csv: {len(page_sample_new)} baris (gabungan, status ter-update)")

MIN_PAGE_TYPES = 4
rendered_mask = page_sample_new['render_status'] == 'rendered'
type_counts = page_sample_new[rendered_mask].groupby('website_id')['page_type'].nunique()

all_website_ids = page_sample_new['website_id'].unique()
now_excluded_ids = [w for w in all_website_ids if w not in type_counts.index or type_counts.get(w, 0) < MIN_PAGE_TYPES]

excluded_new = page_sample_new[page_sample_new['website_id'].isin(now_excluded_ids)].copy()
excluded_new.to_csv('excluded_websites.csv', index=False)
print(f"excluded.csv: {len(now_excluded_ids)} website masih di bawah ambang -> {now_excluded_ids}")

included_ids = [w for w in all_website_ids if w not in now_excluded_ids]
print(f"Website yang lolos sekarang: {len(included_ids)}/{len(all_website_ids)}")

audit_findings.csv: 4062 -> 4109 baris
page_sample_updated.csv: 170 baris (gabungan, status ter-update)
excluded.csv: 4 website masih di bawah ambang -> ['W07', 'W26', 'W28', 'W30']
Website yang lolos sekarang: 30/34


In [ ]:
rendered_counts = page_sample_new[page_sample_new['render_status'] == 'rendered'].groupby('website_id').size()

website_master_new = website_master_old.set_index('website_id').copy()
for wid in included_ids:
    if wid not in website_master_new.index:
        website_master_new.loc[wid] = ''

website_master_new['total_pages_audited'] = website_master_new.index.map(rendered_counts).fillna(0).astype(int)
website_master_new['crawl_status'] = website_master_new['total_pages_audited'].apply(
    lambda n: 'complete' if n >= MIN_PAGE_TYPES else 'incomplete'
)
website_master_new = website_master_new.reset_index()
website_master_new = website_master_new[website_master_new['website_id'].isin(included_ids)]
website_master_new.to_csv('website_mast.csv', index=False)

print(f"website_master.csv: {len(website_master_new)} website")
print(website_master_new['crawl_status'].value_counts())

website_master.csv: 30 website
crawl_status
complete    30
Name: count, dtype: int64


## Catatan

- Baris yang **masih gagal** setelah retry ini (lihat `excluded_websites.csv` yang baru) berarti kemungkinan situsnya memang tidak stabil/memblokir bot — dokumentasikan sebagai alasan eksklusi di naskah (bagian 4.3 guideline: *"dokumentasikan alasan dan gunakan denominator jumlah halaman yang benar-benar berhasil diaudit"*), jangan dipaksa retry berkali-kali.
- Kolom `http_status` masih belum terisi di kedua notebook ini — kalau mau dilengkapi untuk data quality check (bagian 9.1), bilang aja, aku tambahin cell terpisah.
- Setelah semua ini stabil, lanjut ke Minggu 4: deduplikasi temuan + audit ulang subset reproducibility (bagian 6.3).